# Social Transformer - run the finalised model on Colab

Trains the **finalised Social Transformer** for pedestrian trajectory prediction (TrajNet), evaluates it, compares it with
the existing Social LSTM, and produces everything the project needs: **result tables, figures, paired significance tests and a TrajNet submission**.

**Before you start**
1. `Runtime > Change runtime type > T4 GPU` (the code also runs on CPU, but is much slower).
2. Step 1 gets the code **and** the TrajNet text data by cloning this GitHub repository (`REPO_URL` is preset to it).
   Without GitHub access (or for a private repository) use `SOURCE = "upload"` and upload the repository as one zip (GitHub: `Code > Download ZIP`).
3. Run the cells top to bottom. **MODE** decides how long it takes:

| MODE | what it does | purpose |
|---|---|---|
| `smoke` | 1 epoch on a tiny subset | check that everything runs (minutes) |
| `quick` | a few epochs per model | fast test of the results; numbers are slightly worse than the full schedule |
| `full`  | 30 epochs (LSTMs: 8 / 20) | reproduces the numbers of the full study (`RESULTS.md`) |

Metrics: ADE / FDE / **Avg = (ADE+FDE)/2** in metres, lower is better (TrajNet ranking score; the professor's Social LSTM: 1.3865 on the TrajNet test server).
Two protocols: *official* (train -> validation) and *held-out videos* (7 videos never seen in training - closer to the real test).

In [ ]:
#@title Step 1 - get code + data into Colab
import glob, os, subprocess, zipfile

SOURCE = "github"  #@param ["github", "upload"]
REPO_URL = "https://github.com/aakritip23/Deep-Learning-for-Crowd-modelling.git"  #@param {type:"string"}
WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)

if SOURCE == "github":      # clone the repository: code and TrajNet text data are inside it (private repo: https://<token>@github.com/...)
    assert "<your-" not in REPO_URL, "set REPO_URL to the address of this GitHub repository (or use SOURCE = 'upload')"
    if not os.path.isdir(f"{WORK}/repo"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, f"{WORK}/repo"], check=True)
else:                       # upload the whole repository as ONE zip (GitHub: Code > Download ZIP); a file dialog opens
    from google.colab import files
    print("Select the repository zip file")
    for name, content in files.upload().items():
        open(os.path.join(WORK, name), "wb").write(content)
        zipfile.ZipFile(os.path.join(WORK, name)).extractall(WORK)

hits = sorted(glob.glob(f"{WORK}/**/03_improved_model", recursive=True))
assert hits, "folder 03_improved_model not found - is this the project repository?"
PROJECT = hits[0]
DATA = os.path.join(os.path.dirname(PROJECT), "01_original_social_lstm", "social-lstm", "data")
assert os.path.isdir(f"{DATA}/train"), f"TrajNet data not found at {DATA}"
%cd $PROJECT
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (use Runtime > Change runtime type > GPU)")
print("data:", {s: len(glob.glob(f"{DATA}/{s}/*/*.txt")) for s in ("train", "validation", "test")}, "files")

In [ ]:
#@title Step 2 - settings
MODE = "quick"          #@param ["smoke", "quick", "full"]
RUN_COMPARISON = True   #@param {type:"boolean"}
RUN_HOLDOUT = True      #@param {type:"boolean"}
FINAL_FIT = False       #@param {type:"boolean"}
TF_EPOCHS = 0           #@param {type:"integer"}
# RUN_COMPARISON: also train the individual Transformer + the Social LSTM baselines (tables/figures against the existing model).
#                 False = only the finalised model (fastest).
# RUN_HOLDOUT:    also evaluate on 7 videos never seen in training (more trustworthy; costs one extra run per model).
# FINAL_FIT:      additionally fit on train+validation and use that model for the submission (recommended with MODE="full").
# TF_EPOCHS:      0 = the mode's default (smoke 1, quick 8, full 30); e.g. 4 for the absolute minimum.
import torch
if not torch.cuda.is_available() and MODE != "smoke":
    print("WARNING: no GPU - this will be slow. Switch the runtime to GPU or use MODE='smoke' / TF_EPOCHS=2.")

### What to expect
Avg = (ADE+FDE)/2 in metres, lower is better. *Constant velocity* (just extrapolating the walking velocity) scores **1.206** (official) / **1.169** (held-out).
Seed 0 everywhere; a rerun differs by about +/-0.005 (seed noise).

| Model | CPU, 4 epochs | CPU, `quick` (8 epochs) | **Colab GPU, `quick` (8 epochs)** | CPU, `full` (30 epochs) |
|---|---|---|---|---|
| **Social Transformer**, official split | 1.066 | 1.044 | **1.042** | **1.008** |
| **Social Transformer**, held-out videos | - | 1.027 | **1.028** | 1.020 |
| Individual Transformer, official / held-out | - | 1.075 / 1.055 | 1.075 / 1.053 | 1.054 / 1.044 |
| Social LSTM refined (velocity), official | - | 1.127 (8 ep) | 1.123 (8 ep) | 1.105 (20 ep) |
| Social LSTM, professor's settings, official | - | about 2.0 (4 ep) | 1.849 (4 ep) | 2.193 |

The professor-settings LSTM is unstable (its validation error is best after 2-3 epochs and then explodes), so the best epoch is kept.
Even 4 epochs beat constant velocity by 12 %, and 8 epochs already beat the fully trained individual Transformer; on held-out videos 8 epochs are within 0.01 of the full result.
**Fastest possible test of just the final model:** `MODE = "quick"`, `RUN_COMPARISON = False`, `RUN_HOLDOUT = False`, `TF_EPOCHS = 4` (one 4-epoch run + figures + submission).
Runtime on a Colab GPU (measured in the final run): about 5-8 s per Transformer epoch and 26 s per LSTM epoch; the six trainings of the `quick` run took 8.8 minutes in total
(on a CPU one Transformer epoch takes 2-4 minutes).

In [ ]:
#@title Step 3 - run (training log streams below; re-running this cell resumes / skips finished runs)
OUT = "/content/results"
args = f"--data-root {DATA} --out {OUT} --mode {MODE}"
if not RUN_COMPARISON: args += " --no-comparison"
if not RUN_HOLDOUT:    args += " --no-holdout"
if FINAL_FIT:          args += " --final-fit"
if TF_EPOCHS:          args += f" --tf-epochs {TF_EPOCHS}"
!python -u colab_run.py {args}

In [ ]:
#@title Step 4 - show the results
import os
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{OUT}/RESULTS_SUMMARY.md").read()))
for f in ["fig_comparison.png", "fig_curves_official.png", "fig_per_source_official.png",
          "fig_examples_official.png", "fig_examples_holdout.png"]:
    p = f"{OUT}/results/{f}"
    if os.path.exists(p):
        display(Image(p))

In [ ]:
#@title Step 5 - download everything (tables, figures, checkpoints, TrajNet submission)
from google.colab import files
files.download(f"{OUT}/results_bundle.zip")   # contains RESULTS_SUMMARY.md, results/, submission.zip, runs/*/*/best.pt + logs

### What you get (`results_bundle.zip`)
* `RESULTS_SUMMARY.md` - the comparison tables (paste-ready) + the paired significance tests
* `results/fig_comparison.png`, `fig_curves_official.png`, `fig_per_source_official.png`, `fig_examples_*.png` - report figures
* `submission.zip` - the TrajNet test files with every `?` filled in by the model (upload this to the TrajNet server; ground truth is hidden, so there is no local test score)
* `runs/<protocol>/<model>/` - `best.pt` checkpoint, `log.csv` (per-epoch validation), `final_metrics.json`

To keep a run with the project, unzip it into `04_results/<run name>/` of the repository (the run of this repository is in `04_results/colab_final_run/`).

**Reading the table:** *Avg* is the TrajNet score; *vs const. vel.* is Avg divided by the constant-velocity Avg (below 1.00x = better than extrapolating the walking velocity);
*full-study Avg* is the number from the full 30-epoch study - a `quick` run should be close to it but slightly worse.